In [0]:
from pyspark.sql.functions import sum, col, when

In [0]:
storage_account = "economicindiast"
container = "raw-data"

account_key = dbutils.secrets.get(
    scope="dataScope",
    key="sectyu"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    account_key
)

df = spark.read.csv(
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
dbutils.fs.ls("wasbs://raw-data@economicindiast.blob.core.windows.net/")

[FileInfo(path='wasbs://raw-data@economicindiast.blob.core.windows.net/HDI.csv', name='HDI.csv', size=1781357, modificationTime=1790660950000),
 FileInfo(path='wasbs://raw-data@economicindiast.blob.core.windows.net/WorldBank.xlsx.txt', name='WorldBank.xlsx.txt', size=1751538, modificationTime=1790660983000),
 FileInfo(path='wasbs://raw-data@economicindiast.blob.core.windows.net/world_indicators_data_dictionary.csv', name='world_indicators_data_dictionary.csv', size=4138, modificationTime=1790660967000)]

In [0]:
HDI_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("wasbs://raw-data@economicindiast.blob.core.windows.net/HDI.csv")
)

In [0]:
print(spark.version)

4.0.0


In [0]:
print(spark.conf.get(
    "spark.databricks.clusterUsageTags.sparkVersion",
    "unknown"
))

In [0]:
World_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("wasbs://raw-data@economicindiast.blob.core.windows.net/world_indicators_data_dictionary.csv")
)

In [0]:
World_df.show(4)

In [0]:
print(World_df.columns)
print(HDI_df.columns)

In [0]:
HDI_df.show()

In [0]:
HDI_df = HDI_df.withColumnRenamed("country", "country_name")
World_df = World_df.withColumnRenamed("Table", "table").withColumnRenamed("Field", "field").withColumnRenamed("Description", "description")

HDI_df.show(1)

In [0]:
null_counts_HDI_df = HDI_df.select([sum(when(col(column).isNull(), 1).otherwise(0)).alias(column) for column in HDI_df.columns])
null_counts_World_df = World_df.select([sum(when(col(column).isNull(), 1).otherwise(0)).alias(column) for column in World_df.columns])



display(null_counts_HDI_df)
display(null_counts_World_df)


In [0]:
World_df.display()

In [0]:
HDI_df.display()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, LongType, DoubleType, FloatType, DecimalType


string_columns = [
    field.name
    for field in HDI_df.schema.fields
    if isinstance(field.dataType, StringType)
]


numeric_columns = [
    field.name
    for field in HDI_df.schema.fields
    if isinstance(field.dataType, (IntegerType, LongType, DoubleType, FloatType, DecimalType))
]


HDI_df = HDI_df.fillna(
    "Unknown",
    subset=string_columns
)


HDI_df = HDI_df.fillna(
    0,
    subset=numeric_columns
)

HDI_df.show()

In [0]:
HDI_df.display()

In [0]:
HDI_df.write.option("headers","true").csv("wasbs://transformed-data@economicindiast.blob.core.windows.net/HDI.csv")
World_df.write.option("headers","true").csv("wasbs://transformed-data@economicindiast.blob.core.windows.net/World.csv")